# NASA C-MAPSS FD001 — Advanced RUL Benchmark

**Exploratory model development · Ridge, tree ensembles, LSTM, GRU, feature engineering and safety-relevant error analysis**

NASA C-MAPSS FD001 is a **simulated** turbofan run-to-failure dataset. We use an engine-disjoint development split (80 training / 20 validation engines). Reported metrics below are **validation results, not official FD001 test scores**. The separate official test set is not used for model selection here.

**Objective:** predict the *uncapped* remaining useful life (RUL) using only sensor observations available up to prediction time. Compare overall accuracy, balanced engine accuracy and overestimation near failure. All examples are exploratory; repeated model selection on the same validation engines can make results optimistic.

**Reproduction:** Attach Kaggle dataset `behrad3d/nasa-cmaps`, then run the notebook top to bottom. Neural-network results can vary slightly with hardware and software versions. This notebook trains several models and may take several minutes, especially on CPU.

## 1. Data loading and engine-disjoint split

RUL for each complete *training* trajectory is `last_cycle - current_cycle`. The final cycle and RUL are **targets/metadata**, never input features.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps")

columns = (
    ["engine_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

train["max_cycle"] = train.groupby("engine_id")["cycle"].transform("max")
train["RUL"] = train["max_cycle"] - train["cycle"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(train, groups=train["engine_id"])
)

train_df = train.iloc[train_idx].copy()
val_df = train.iloc[val_idx].copy()

print("Training engines:", train_df.engine_id.nunique())
print("Validation engines:", val_df.engine_id.nunique())
print("Training observations:", len(train_df))
print("Validation observations:", len(val_df))
print("Available sensor features:", 21)


## 2. Training-only feature screening

Start with 21 sensors and three settings; remove constant features using training engines only (17 features remain).

In [ ]:
# 21 sensors + 3 operating settings
candidate_features = (
    [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

# Calculate variation using training engines ONLY
feature_std = train_df[candidate_features].std()

# Remove constant and near-constant features
selected_features = feature_std[
    feature_std > 1e-8
].index.tolist()

removed_features = [
    f for f in candidate_features
    if f not in selected_features
]

# Prepare common input and target arrays
X_train = train_df[selected_features]
X_val = val_df[selected_features]

y_train = train_df["RUL"]
y_val = val_df["RUL"]

print("Candidate features:", len(candidate_features))
print("Selected features:", len(selected_features))
print("Removed features:", removed_features)
print("Missing training values:", X_train.isna().sum().sum())
print("Missing validation values:", X_val.isna().sum().sum())


## 3. Traditional ML baselines

Ridge learns a regularised linear relationship; Random Forest averages tree predictions; gradient boosting builds a sequence of corrective trees. Their inputs are current-cycle features only. Metrics cover **all validation cycles**, not one prediction per engine.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

models = {
    "Ridge Regression": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        max_leaf_nodes=15,
        learning_rate=0.05,
        l2_regularization=1.0,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    trained_models[name] = model

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
        "R2": r2_score(y_val, predictions),
        "Bias": np.mean(predictions - y_val),
        "Overestimation_%": 100 * np.mean(predictions > y_val)
    })

    print(f"Completed: {name}")

results_df = pd.DataFrame(results).sort_values("MAE")

display(results_df.round(3))


## 4. Temporal windows for neural networks

Each sequence contains the current cycle and up to 29 preceding cycles, left-padded by the earliest observation for short histories. The scaler is fitted only on training engines.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

WINDOW = 30

scaler = StandardScaler()
scaler.fit(train_df[selected_features])

def build_sequences(data, features, scaler, window=30):
    sequences, targets, engine_ids = [], [], []

    for engine_id, group in data.groupby("engine_id"):
        group = group.sort_values("cycle")

        values = scaler.transform(group[features])
        targets_engine = group["RUL"].to_numpy()

        padding = np.repeat(
            values[:1], window - 1, axis=0
        )
        padded = np.vstack([padding, values])

        for i in range(len(values)):
            sequences.append(padded[i:i + window])
            targets.append(targets_engine[i])
            engine_ids.append(engine_id)

    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(targets, dtype=np.float32),
        np.asarray(engine_ids)
    )

X_train_seq, y_train_seq, train_engine_ids = build_sequences(
    train_df, selected_features, scaler, WINDOW
)

X_val_seq, y_val_seq, val_engine_ids = build_sequences(
    val_df, selected_features, scaler, WINDOW
)

print("Training sequences:", X_train_seq.shape)
print("Validation sequences:", X_val_seq.shape)
print("Training targets:", y_train_seq.shape)
print("Validation targets:", y_val_seq.shape)


## 5. LSTM — gated sequence model

LSTM processes 30 consecutive measurements with memory gates. Early stopping monitors validation MSE and restores the best weights.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

lstm_model = Sequential([
    Input(shape=(30, 17)),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=60,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
lstm_predictions = lstm_model.predict(
    X_val_seq, verbose=0
).ravel()

lstm_results = {
    "Model": "LSTM",
    "MAE": mean_absolute_error(y_val_seq, lstm_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_val_seq, lstm_predictions)),
    "R2": r2_score(y_val_seq, lstm_predictions),
    "Bias": np.mean(lstm_predictions - y_val_seq),
    "Overestimation_%": 100 * np.mean(lstm_predictions > y_val_seq)
}

display(pd.DataFrame([lstm_results]).round(3))

print("Epochs trained:", len(history.history["loss"]))
print("Best validation epoch:",
      np.argmin(history.history["val_loss"]) + 1)


## 6. GRU — compact gated sequence model

GRU uses a simpler gating mechanism than LSTM. Compare identical inputs, targets, split and training settings.

In [ ]:
from tensorflow.keras.layers import GRU

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

gru_model = Sequential([
    Input(shape=(30, 17)),
    GRU(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

gru_model.summary()

gru_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

gru_history = gru_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=60,
    batch_size=128,
    callbacks=[gru_stop],
    verbose=1
)


In [ ]:
gru_predictions = gru_model.predict(X_val_seq, verbose=0).ravel()

gru_results = {
    "Model": "GRU",
    "MAE": mean_absolute_error(y_val_seq, gru_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_val_seq, gru_predictions)),
    "R2": r2_score(y_val_seq, gru_predictions),
    "Bias": np.mean(gru_predictions - y_val_seq),
    "Overestimation_%": 100 * np.mean(gru_predictions > y_val_seq)
}

comparison_df = pd.concat(
    [results_df, pd.DataFrame([lstm_results, gru_results])],
    ignore_index=True
).sort_values("MAE")

display(comparison_df.round(3))

print("GRU epochs:", len(gru_history.history["loss"]))
print("GRU best epoch:", np.argmin(gru_history.history["val_loss"]) + 1)
print("GRU parameters:", gru_model.count_params())
print("LSTM parameters:", lstm_model.count_params())


## 7. Capped RUL target: a different prediction objective

Capping target values at 125 simplifies distant-life prediction but **changes the task**. Compare any candidate to the original uncapped labels for full-range RUL. Do not mistake capped-target MAE for full-range MAE.

In [ ]:
CAP = 125

train_capped = np.minimum(y_train_seq, CAP)
val_capped = np.minimum(y_val_seq, CAP)

print("Training sequences:", len(y_train_seq))

print(
    "Training targets above cap:",
    int((y_train_seq > CAP).sum())
)

print(
    "Training percentage capped:",
    round(100 * (y_train_seq > CAP).mean(), 2), "%"
)

print(
    "Validation percentage above cap:",
    round(100 * (y_val_seq > CAP).mean(), 2), "%"
)

print(
    "Original maximum RUL:",
    y_train_seq.max()
)

print(
    "Capped maximum RUL:",
    train_capped.max()
)


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, GRU, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

gru_capped = Sequential([
    Input(shape=(30, 17)),
    GRU(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

gru_capped.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

stop_capped = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_capped = gru_capped.fit(
    X_train_seq,
    train_capped,
    validation_data=(X_val_seq, val_capped),
    epochs=60,
    batch_size=128,
    callbacks=[stop_capped],
    verbose=1
)


In [ ]:
pred_capped = gru_capped.predict(X_val_seq, verbose=0).ravel()

print("MAE against capped RUL:",
      round(mean_absolute_error(val_capped, pred_capped), 3))

print("MAE against original RUL:",
      round(mean_absolute_error(y_val_seq, pred_capped), 3))

print("RMSE against original RUL:",
      round(np.sqrt(mean_squared_error(y_val_seq, pred_capped)), 3))

print("Bias against original RUL:",
      round(np.mean(pred_capped - y_val_seq), 3))


## 8. GRU window ablation: 15, 30 and 60 cycles

Change the input history length while holding the training protocol constant. The original 30-cycle baseline is evaluated from its stored model, rather than typed in manually.

In [ ]:
sequence_results = [{
    "Window": 30,
    "MAE": float(gru_results["MAE"]),
    "RMSE": float(gru_results["RMSE"]),
    "Source": "Previous GRU"
}]

sequence_models = {}

for window in [15, 60]:

    print(f"\nTraining GRU with {window} cycles")

    Xtr, ytr, _ = build_sequences(
        train_df, selected_features, scaler, window
    )

    Xvl, yvl, _ = build_sequences(
        val_df, selected_features, scaler, window
    )

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = Sequential([
        Input(shape=(window, len(selected_features))),
        GRU(64),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="mse",
        metrics=["mae"]
    )

    stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history_window = model.fit(
        Xtr, ytr,
        validation_data=(Xvl, yvl),
        epochs=60,
        batch_size=128,
        callbacks=[stop],
        verbose=0
    )

    pred = model.predict(Xvl, verbose=0).ravel()

    sequence_results.append({
        "Window": window,
        "MAE": mean_absolute_error(yvl, pred),
        "RMSE": np.sqrt(mean_squared_error(yvl, pred)),
        "Source": "New experiment"
    })

    sequence_models[window] = model

    print(
        f"Completed: {window} cycles, "
        f"MAE = {sequence_results[-1]['MAE']:.3f}"
    )

sequence_comparison = pd.DataFrame(
    sequence_results
).sort_values("Window")

display(sequence_comparison.round(3))


## 9. Endpoint diagnostics and engine-balanced validation

All-cycle errors give longer trajectories greater weight. Fixed operating-age evaluations give one prediction per eligible engine; engine-balanced MAE averages each engine’s within-trajectory MAE equally. These are **development diagnostics**, not independent test results.

In [ ]:
# Identify cycle-100 positions in validation data
val_cycle100 = val_df.reset_index(drop=True)["cycle"].eq(100).to_numpy()

assert val_cycle100.sum() == 20

actual = y_val.to_numpy()[val_cycle100]

endpoint_results = []

# Traditional ML models
for name, model in trained_models.items():
    predictions = model.predict(X_val)[val_cycle100]

    endpoint_results.append({
        "Model": name,
        "MAE": mean_absolute_error(actual, predictions),
        "RMSE": np.sqrt(mean_squared_error(actual, predictions)),
        "Bias": np.mean(predictions - actual)
    })

# Neural-network models
for name, model in [
    ("LSTM", lstm_model),
    ("GRU", gru_model)
]:
    predictions = model.predict(
        X_val_seq, verbose=0
    ).ravel()[val_cycle100]

    endpoint_results.append({
        "Model": name,
        "MAE": mean_absolute_error(actual, predictions),
        "RMSE": np.sqrt(mean_squared_error(actual, predictions)),
        "Bias": np.mean(predictions - actual)
    })

endpoint_df = pd.DataFrame(endpoint_results).sort_values("MAE")

display(endpoint_df.round(3))


In [ ]:
ages = [50, 75, 100, 125, 150]

# Calculate predictions once
all_predictions = {
    name: model.predict(X_val)
    for name, model in trained_models.items()
}

all_predictions["LSTM"] = lstm_model.predict(
    X_val_seq, verbose=0
).ravel()

all_predictions["GRU"] = gru_model.predict(
    X_val_seq, verbose=0
).ravel()

# Verify that predictions refer to the same validation rows
assert np.array_equal(y_val.to_numpy(), y_val_seq)
assert np.array_equal(
    val_df["engine_id"].to_numpy(),
    val_engine_ids
)

rows = []

for age in ages:
    mask = val_df["cycle"].to_numpy() == age
    actual = y_val.to_numpy()[mask]

    for name, preds in all_predictions.items():
        selected = preds[mask]
        errors = selected - actual

        rows.append({
            "Cycle": age,
            "Model": name,
            "Engines": len(actual),
            "MAE": np.mean(np.abs(errors)),
            "Bias": np.mean(errors),
            "Overestimation_%": 100 * np.mean(errors > 0)
        })

age_results = pd.DataFrame(rows)

display(
    age_results.pivot(
        index="Cycle",
        columns="Model",
        values="MAE"
    ).round(2)
)

print("\nEngines available at each age:")
display(age_results.groupby("Cycle")["Engines"].first())


In [ ]:
# Prediction arrays were created in the previous step
engine_results = []

engine_ids = val_df["engine_id"].to_numpy()
actual = y_val.to_numpy()

for name, preds in all_predictions.items():

    errors = np.abs(preds - actual)

    temp = pd.DataFrame({
        "engine_id": engine_ids,
        "absolute_error": errors
    })

    per_engine_mae = (
        temp.groupby("engine_id")["absolute_error"]
        .mean()
    )

    engine_results.append({
        "Model": name,
        "Engine_Balanced_MAE": per_engine_mae.mean(),
        "Std_Across_Engines": per_engine_mae.std(),
        "Worst_Engine_MAE": per_engine_mae.max()
    })

engine_comparison = pd.DataFrame(
    engine_results
).sort_values("Engine_Balanced_MAE")

display(engine_comparison.round(3))


## 10. Sensitivity to random initialisation

Same split and architecture, different random seeds. This checks training variability, **not** variability across held-out engine groups.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

seeds = [7, 21, 123]
seed_results = []

for seed in seeds:

    print(f"\nTraining GRU with seed {seed}")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(30, 17)),
        tf.keras.layers.GRU(64),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="mse",
        metrics=["mae"]
    )

    stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history_seed = model.fit(
        X_train_seq,
        y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=60,
        batch_size=128,
        callbacks=[stop],
        verbose=0
    )

    predictions = model.predict(
        X_val_seq, verbose=0
    ).ravel()

    abs_errors = np.abs(predictions - y_val_seq)

    per_engine = pd.DataFrame({
        "engine_id": val_engine_ids,
        "abs_error": abs_errors
    }).groupby("engine_id")["abs_error"].mean()

    seed_results.append({
        "Seed": seed,
        "MAE": abs_errors.mean(),
        "Engine_Balanced_MAE": per_engine.mean(),
        "Best_Epoch": np.argmin(
            history_seed.history["val_loss"]
        ) + 1
    })

    print(
        f"MAE = {abs_errors.mean():.3f}, "
        f"Balanced MAE = {per_engine.mean():.3f}"
    )

seed_df = pd.DataFrame(seed_results)

# Add the previously recorded seed-42 experiment
seed_df = pd.concat([
    seed_df,
    pd.DataFrame([{
        "Seed": 42,
        "MAE": float(gru_results["MAE"]),
        "Engine_Balanced_MAE": float(engine_comparison.loc[engine_comparison["Model"] == "GRU", "Engine_Balanced_MAE"].iloc[0]),
        "Best_Epoch": int(np.argmin(gru_history.history["val_loss"]) + 1)
    }])
], ignore_index=True).sort_values("Seed")

display(seed_df.round(3))

print("\nMean MAE:", round(seed_df["MAE"].mean(), 3))
print("MAE standard deviation:",
      round(seed_df["MAE"].std(), 3))


## 11. Learning-rate scheduler ablation

Allow Adam to reduce its learning rate when validation loss plateaus; compare the restored best-epoch weights to the original GRU.

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

gru_adaptive = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, 17)),
    tf.keras.layers.GRU(64),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

gru_adaptive.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True
    )
]

adaptive_history = gru_adaptive.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=80,
    batch_size=128,
    callbacks=callbacks,
    verbose=0
)

adaptive_pred = gru_adaptive.predict(
    X_val_seq, verbose=0
).ravel()

print("Adaptive GRU MAE:",
      round(mean_absolute_error(y_val_seq, adaptive_pred), 3))

print("Adaptive GRU RMSE:",
      round(np.sqrt(mean_squared_error(y_val_seq, adaptive_pred)), 3))

print("Adaptive GRU bias:",
      round(np.mean(adaptive_pred - y_val_seq), 3))

print("Best epoch:",
      np.argmin(adaptive_history.history["val_loss"]) + 1)


## 12. Engine-relative temporal features (34-feature GRU)

For each engine, use the mean of the first five available readings as a baseline; for cycles 1–4, use only readings available so far. Append relative changes to 17 original features. This also increases model parameter count, so the effect of features alone is not isolated.

In [ ]:
def add_relative_features(data, features):

    data = data.sort_values(
        ["engine_id", "cycle"]
    ).copy()

    relative_features = [
        f"{feature}_relative" for feature in features
    ]

    for engine_id, group in data.groupby("engine_id"):

        # Sensor measurements for this engine
        values = group[features].to_numpy(dtype=float)

        # Causal baseline: running mean during first five cycles,
        # then fixed at the mean of those five readings.
        baseline = np.empty_like(values)

        for i in range(len(values)):
            n = min(i + 1, 5)
            baseline[i] = values[:n].mean(axis=0)

        relative_values = values - baseline

        data.loc[group.index, relative_features] = relative_values

    return data, relative_features


train_relative, relative_features = add_relative_features(
    train_df, selected_features
)

val_relative, _ = add_relative_features(
    val_df, selected_features
)

enhanced_features = selected_features + relative_features

print("Relative features:", len(relative_features))
print("Total enhanced features:", len(enhanced_features))

# ------------------------------------------------------------
# 6. Standardisation — training engines only
# ------------------------------------------------------------

enhanced_scaler = StandardScaler()

enhanced_scaler.fit(train_relative[enhanced_features])

# ------------------------------------------------------------
# 7. Construct 30-cycle sequences
# ------------------------------------------------------------

X_train_enhanced, y_train_enhanced, train_ids = build_sequences(
    train_relative, enhanced_features, enhanced_scaler, WINDOW
)

X_val_enhanced, y_val_enhanced, val_ids = build_sequences(
    val_relative, enhanced_features, enhanced_scaler, WINDOW
)

print("Training sequence shape:", X_train_enhanced.shape)
print("Validation sequence shape:", X_val_enhanced.shape)

assert X_train_enhanced.shape == (16561, 30, 34)
assert X_val_enhanced.shape == (4070, 30, 34)
assert not np.isnan(X_train_enhanced).any()
assert not np.isnan(X_val_enhanced).any()

# ------------------------------------------------------------
# 8. Train enhanced GRU
# ------------------------------------------------------------

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = Sequential([
    Input(shape=(WINDOW, len(enhanced_features))),
    GRU(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train_enhanced,
    y_train_enhanced,
    validation_data=(X_val_enhanced, y_val_enhanced),
    epochs=60,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

# ------------------------------------------------------------
# 9. Evaluate against ORIGINAL uncapped RUL
# ------------------------------------------------------------

predictions = model.predict(X_val_enhanced, verbose=0).ravel()

mae = mean_absolute_error(y_val_enhanced, predictions)
rmse = np.sqrt(mean_squared_error(y_val_enhanced, predictions))
r2 = r2_score(y_val_enhanced, predictions)
bias = np.mean(predictions - y_val_enhanced)
overestimation = 100 * np.mean(predictions > y_val_enhanced)

print("\n========== ENHANCED GRU RESULTS ==========")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")
print(f"Bias: {bias:.3f}")
print(f"Overestimation: {overestimation:.2f}%")
print(f"Best epoch: {np.argmin(history.history['val_loss']) + 1}")
print(f"Trainable parameters: {model.count_params():,}")
print("==========================================")

print(f"\nOriginal GRU reference MAE: {gru_results['MAE']:.3f} cycles")
print(f"Enhanced GRU MAE: {mae:.3f} cycles")
print(f"MAE improvement: {float(gru_results['MAE']) - mae:+.3f} cycles")


## 13. L2 weight penalty on enhanced GRU

L2 discourages large weights. Any improvement must be judged on **uncapped prediction errors**, not the regularised Keras loss.

In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, GRU, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Confirm we have the enhanced 34-feature data
assert X_train_enhanced.shape[1:] == (30, 34)
assert X_val_enhanced.shape[1:] == (30, 34)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

gru_l2 = Sequential([
    Input(shape=(30, 34)),

    GRU(
        64,
        kernel_regularizer=l2(0.001),
        recurrent_regularizer=l2(0.001)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    ),

    Dense(1)
])

gru_l2.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

stop_l2 = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history_l2 = gru_l2.fit(
    X_train_enhanced,
    y_train_enhanced,
    validation_data=(X_val_enhanced, y_val_enhanced),
    epochs=60,
    batch_size=128,
    callbacks=[stop_l2],
    verbose=0
)

pred_l2 = gru_l2.predict(X_val_enhanced, verbose=0).ravel()

print("====== L2 GRU RESULTS ======")
print("MAE:", round(mean_absolute_error(y_val_enhanced, pred_l2), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val_enhanced, pred_l2)), 3))
print("R2:", round(r2_score(y_val_enhanced, pred_l2), 3))
print("Bias:", round(np.mean(pred_l2 - y_val_enhanced), 3))
print("Best epoch:", np.argmin(history_l2.history["val_loss"]) + 1)
print("Epochs trained:", len(history_l2.history["loss"]))


## 14. Engine-by-engine error audit

Improvement in mean MAE does not imply improvement for every engine. Compare the same validation samples.

In [ ]:
# Compare the two enhanced GRU models on identical validation rows

error_comparison = pd.DataFrame({
    "engine_id": val_ids,
    "actual_RUL": y_val_enhanced,
    "enhanced_error": np.abs(predictions - y_val_enhanced),
    "l2_error": np.abs(pred_l2 - y_val_enhanced)
})

engine_errors = (
    error_comparison.groupby("engine_id")
    [["enhanced_error", "l2_error"]]
    .mean()
)

engine_errors["Improvement"] = (
    engine_errors["enhanced_error"]
    - engine_errors["l2_error"]
)

print(
    "Engines improved by L2:",
    (engine_errors["Improvement"] > 0).sum(),
    "out of",
    len(engine_errors)
)

display(
    engine_errors.sort_values(
        "Improvement", ascending=False
    ).round(2)
)


## 15. Example trajectories — Engines 13 and 84

Diagnostic plots show model-specific prediction patterns; they do not prove which sensor caused a prediction error.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Confirm predictions align with the validation sequences
assert len(predictions) == len(pred_l2) == len(y_val_enhanced) == len(val_ids)

# Reconstruct the original validation-row order
validation_rows = (
    val_relative
    .sort_values(["engine_id", "cycle"])
    .reset_index(drop=True)
)

assert np.array_equal(
    validation_rows["engine_id"].to_numpy(),
    val_ids
)

assert np.allclose(
    validation_rows["RUL"].to_numpy(),
    y_val_enhanced
)

validation_rows["GRU_pred"] = predictions
validation_rows["GRU_L2_pred"] = pred_l2

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for row, engine_id in enumerate([13, 84]):

    engine = validation_rows[
        validation_rows["engine_id"] == engine_id
    ].sort_values("cycle")

    # Left: actual RUL and model predictions
    axes[row, 0].plot(
        engine["cycle"], engine["RUL"],
        label="Actual RUL", linewidth=2
    )

    axes[row, 0].plot(
        engine["cycle"], engine["GRU_pred"],
        label="GRU without L2", alpha=0.8
    )

    axes[row, 0].plot(
        engine["cycle"], engine["GRU_L2_pred"],
        label="GRU with L2", alpha=0.8
    )

    axes[row, 0].set_title(f"Engine {engine_id}: RUL prediction")
    axes[row, 0].set_ylabel("RUL (cycles)")
    axes[row, 0].legend()
    axes[row, 0].grid(alpha=0.3)

    # Right: actual sensor behaviour
    axes[row, 1].plot(
        engine["cycle"], engine["sensor_11"],
        label="Sensor 11"
    )

    axes[row, 1].set_title(f"Engine {engine_id}: Sensor 11")
    axes[row, 1].set_ylabel("Sensor reading")
    axes[row, 1].legend()
    axes[row, 1].grid(alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel("Operating cycle")

plt.tight_layout()
plt.show()


## 16. Cost-sensitive asymmetric loss

Penalise RUL overestimation twice as strongly as underestimation. This is an **illustrative preference**, not a validated maintenance cost ratio. Evaluate overall error and near-failure (actual RUL ≤ 30) diagnostics.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, GRU, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# Ensure we're using the enhanced experiment
assert X_train_enhanced.shape[1:] == (30, 34)
assert X_val_enhanced.shape[1:] == (30, 34)

# Positive error means predicted RUL > actual RUL
def asymmetric_mse(y_true, y_pred):
    error = y_pred - y_true
    penalty = tf.where(error > 0, 2.0, 1.0)
    return tf.reduce_mean(penalty * tf.square(error))

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

gru_asymmetric = Sequential([
    Input(shape=(30, 34)),
    GRU(
        64,
        kernel_regularizer=l2(0.001),
        recurrent_regularizer=l2(0.001)
    ),
    Dropout(0.2),
    Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    ),
    Dense(1)
])

gru_asymmetric.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=asymmetric_mse,
    metrics=["mae"]
)

asym_history = gru_asymmetric.fit(
    X_train_enhanced,
    y_train_enhanced,
    validation_data=(X_val_enhanced, y_val_enhanced),
    epochs=60,
    batch_size=128,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True
        )
    ],
    verbose=0
)

pred_asym = gru_asymmetric.predict(X_val_enhanced, verbose=0).ravel()

def evaluate_predictions(name, pred):
    error = pred - y_val_enhanced
    near_failure = y_val_enhanced <= 30

    return {
        "Model": name,
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "Bias": np.mean(error),
        "Overestimation_%": 100 * np.mean(error > 0),
        "Near_Failure_MAE": np.mean(np.abs(error[near_failure])),
        "Near_Failure_Over_%": 100 * np.mean(error[near_failure] > 0)
    }

asym_comparison = pd.DataFrame([
    evaluate_predictions("Standard L2 GRU", pred_l2),
    evaluate_predictions("Asymmetric L2 GRU", pred_asym)
])

display(asym_comparison.round(3))

print(
    "Best asymmetric epoch:",
    np.argmin(asym_history.history["val_loss"]) + 1
)


In [ ]:
near_failure = y_val_enhanced <= 30

for name, pred in [
    ("Standard L2 GRU", pred_l2),
    ("Asymmetric L2 GRU", pred_asym)
]:
    errors = pred[near_failure] - y_val_enhanced[near_failure]
    positive_errors = errors[errors > 0]

    print(f"\n{name}")
    print("Near-failure observations:", len(errors))
    print("Mean overestimation when positive:",
          round(positive_errors.mean(), 3))
    print("Worst overestimation:",
          round(positive_errors.max(), 3))


## 17. Conclusions and interpretation

**Original recorded validation findings** (results can vary slightly on rerun):

| Experiment | MAE (cycles) | Interpretation |
|---|---:|---|
| Ridge (17 features) | 30.111 | Linear baseline |
| Random Forest (17 features) | 25.724 | Nonlinear tabular baseline |
| Gradient Boosting (17 features) | 25.790 | Sequential tree ensemble |
| LSTM (30 × 17) | 21.562 | Sequence information useful |
| GRU (30 × 17) | 20.843 | Strong initial validation model |
| GRU capped at 125, evaluated on uncapped RUL | 28.278 | Capping worsened full-range task |
| GRU with 34 relative/raw inputs | 20.576 | Slight improvement in MAE |
| Enhanced GRU + L2 | **20.184** | Lower overall MAE; increased underprediction bias |
| Enhanced GRU + L2 + asymmetric loss | 20.862 | Worse overall MAE, improved near-failure MAE |

**Near-failure insight:** for actual RUL ≤ 30 cycles, reported MAE changed from **9.980** (standard L2) to **6.948** (asymmetric L2), and overestimation frequency from **80.00%** to **60.32%**. These are retrospective cycle-level diagnostics from the same 20 held-out engines; they are not operational safety guarantees.

**Limitations:** One engine-disjoint 80/20 split was reused for many exploratory choices. Adjacent cycle observations are correlated. Hardware and library versions may affect training determinism. GRU and LSTM had different input modalities from tabular baselines (30 cycles vs current cycle). Selected validation scores can be optimistic. There is **no new official-test result** for this advanced notebook.

**Future work:** Repeated engine-disjoint folds, repeated seeds for selected model pairs, a locked endpoint-focused selection protocol, then carefully controlled test reporting without repeatedly tuning to test labels. Do not publish the exploratory validation score as official test performance.